# Unseen-household holdout design

This notebook freezes a household-level holdout before model training. It samples 20% of eligible households that have both outer-development rows and future outer-test rows. Eligible households are stratified by their predominant pre-test branch and pre-test observed-history length. Future P95/P90 outcomes are used only after the assignment is frozen, to audit test support; they do not influence selection.

Household IDs assigned to `intentional_holdout` must be excluded from model fitting, calibration, and threshold selection in later notebooks. Their historical meter measurements may still supply lagged features for their future test rows. The naturally unseen outer-test cohort remains separately identified.


## Stage 0 — inputs and provenance

The master split from Notebook 04 defines the outer dates and internal development blocks. A candidate must have an eligible model-ready outer-test row and a household record in outer development. Historical measurement availability and branch are summarized only from dates before the outer test starts.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir() or not (ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {ROOT}")
DATA_DIR = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "reports" / "tables" / "04b_unseen_household_holdout_design"
FIGURE_DIR = ROOT / "reports" / "figures" / "04b_unseen_household_holdout_design"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_DIR / "04_master_split.csv"
MASTER_CONTRACT_PATH = DATA_DIR / "04_master_split_dtype_contract.json"
SPLIT_MANIFEST_PATH = ROOT / "reports" / "tables" / "04_split_design" / "04_split_design_manifest.json"
P95_PATH = DATA_DIR / "05_model_ready_m0_p95.csv"
P95_CONTRACT_PATH = DATA_DIR / "05_model_ready_m0_p95_dtype_contract.json"
P90_PATH = DATA_DIR / "05_model_ready_m0_p90.csv"
P90_CONTRACT_PATH = DATA_DIR / "05_model_ready_m0_p90_dtype_contract.json"
OBSERVED_PATH = DATA_DIR / "02_observed_household_day.csv"
OBSERVED_CONTRACT_PATH = DATA_DIR / "02_observed_household_day_dtype_contract.json"
ASSIGNMENT_PATH = DATA_DIR / "04b_household_holdout_assignment.csv"
ASSIGNMENT_CONTRACT_PATH = DATA_DIR / "04b_household_holdout_assignment_dtype_contract.json"

HOLDOUT_FRACTION = 0.20
RANDOM_SEED = 20261007
MIN_PRETEST_OBSERVED_DAYS = 45
MIN_HOLDOUT_TEST_EVENTS = 100

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def read_contract_csv(path, contract_path, usecols=None):
    contract = read_json(contract_path)
    header = pd.read_csv(path, nrows=0, encoding=contract["encoding"], sep=contract["separator"]).columns.tolist()
    if header != list(contract["columns"]):
        raise RuntimeError(f"CSV header differs from its dtype contract: {path.name}")
    columns = list(contract["columns"]) if usecols is None else list(usecols)
    dates = [name for name in contract.get("date_columns", []) if name in columns]
    dtypes = {name: dtype for name, dtype in contract["columns"].items() if name in columns and dtype != "datetime"}
    reader = contract["reader"]
    return pd.read_csv(path, usecols=columns, dtype=dtypes or None, parse_dates=dates or None,
                       true_values=reader["true_values"], false_values=reader["false_values"],
                       keep_default_na=reader["keep_default_na"], na_values=reader["na_values"],
                       encoding=contract["encoding"], sep=contract["separator"], low_memory=False)

def write_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def write_dtype_contract(frame, path, contract_path):
    def dtype_name(dtype):
        if pd.api.types.is_datetime64_any_dtype(dtype): return "datetime"
        if pd.api.types.is_bool_dtype(dtype): return "boolean"
        if pd.api.types.is_integer_dtype(dtype): return str(dtype)
        if pd.api.types.is_float_dtype(dtype): return str(dtype)
        return "string"
    columns = {name: dtype_name(frame[name].dtype) for name in frame.columns}
    contract = {"contract_version": 1, "path": str(path), "format": "csv", "encoding": "utf-8", "separator": ",",
                "missing_value": "", "date_columns": [name for name, dtype in columns.items() if dtype == "datetime"],
                "columns": columns, "reader": {"true_values": ["True"], "false_values": ["False"],
                "keep_default_na": False, "na_values": [""]}}
    write_json(contract, contract_path)
    return contract

required = [MASTER_PATH, MASTER_CONTRACT_PATH, SPLIT_MANIFEST_PATH, P95_PATH, P95_CONTRACT_PATH,
            P90_PATH, P90_CONTRACT_PATH, OBSERVED_PATH, OBSERVED_CONTRACT_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Unseen-household design inputs are missing: {missing}")

split_manifest = read_json(SPLIT_MANIFEST_PATH)
master_signature = split_manifest["master_split"]
if sha256(MASTER_PATH) != master_signature["sha256"] or MASTER_PATH.stat().st_size != master_signature.get("size_bytes", MASTER_PATH.stat().st_size):
    raise RuntimeError("Notebook 04 master split does not match its frozen manifest")
master = read_contract_csv(MASTER_PATH, MASTER_CONTRACT_PATH)
master["household_id"] = master["household_id"].astype("string")
master["target_date"] = pd.to_datetime(master["target_date"], errors="raise").dt.normalize()
if not master.row_id.is_unique or not master.household_id.notna().all():
    raise RuntimeError("Master split must have unique row IDs and non-missing household IDs")

P95_KEYS = ["row_id", "household_id", "measurement_date"]
p95 = read_contract_csv(P95_PATH, P95_CONTRACT_PATH, P95_KEYS)
p90 = read_contract_csv(P90_PATH, P90_CONTRACT_PATH, P95_KEYS)
for frame in (p95, p90):
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()
if not p95["row_id"].equals(p90["row_id"]) or not p95["household_id"].equals(p90["household_id"]):
    raise RuntimeError("P95 and P90 row populations are not identically ordered")
if set(master.row_id) != set(p95.row_id):
    raise RuntimeError("Master split and model-ready rows do not have the same row IDs")

outer_test = master.loc[master.outer_split.eq("test")].copy()
if outer_test.empty:
    raise RuntimeError("The frozen outer test period is empty")
TEST_START = outer_test.target_date.min()
observed = read_contract_csv(OBSERVED_PATH, OBSERVED_CONTRACT_PATH,
                             ["household_id", "measurement_date", "branch", "day_observed"])
observed["household_id"] = observed["household_id"].astype("string")
observed["measurement_date"] = pd.to_datetime(observed["measurement_date"], errors="raise").dt.normalize()
observed_pretest = observed.loc[observed.measurement_date.lt(TEST_START)].copy()
print(f"Outer test begins {TEST_START.date()}; master rows={len(master):,}; test rows={len(outer_test):,}.")
print("Frozen master split SHA-256:", master_signature["sha256"])


## Stage 1 — define eligible households and pre-test strata

The candidate pool contains households with both outer-development rows and outer-test rows. A test row in the model-ready datasets confirms the household has the history needed by the frozen feature pipeline. We also audit observed pre-test days directly. Branch and history length are descriptive strata only; no target values are used to form them.


In [ ]:
development_households = set(master.loc[master.outer_split.eq("development"), "household_id"].astype(str))
test_seen_households = set(outer_test.loc[outer_test.test_cohort.eq("test_seen"), "household_id"].astype(str))
natural_unseen_households = set(outer_test.loc[outer_test.test_cohort.eq("test_unseen"), "household_id"].astype(str))
eligible_ids = sorted(development_households & test_seen_households)
if not eligible_ids:
    raise RuntimeError("No households have both outer-development and future test rows")

history_summary = observed_pretest.groupby("household_id", as_index=False).agg(
    pretest_observed_days=("measurement_date", "nunique"),
    predominant_branch=("branch", lambda values: values.dropna().astype(str).mode().sort_values().iloc[0]
                       if not values.dropna().empty else "Unknown"))
history_summary["household_id"] = history_summary["household_id"].astype("string")
candidate_pool = pd.DataFrame({"household_id": pd.Series(eligible_ids, dtype="string")}).merge(
    history_summary, on="household_id", how="left", validate="one_to_one")
candidate_pool["pretest_observed_days"] = candidate_pool["pretest_observed_days"].fillna(0).astype("int32")
candidate_pool["predominant_branch"] = candidate_pool["predominant_branch"].fillna("Unknown").astype(str)

if candidate_pool["pretest_observed_days"].lt(MIN_PRETEST_OBSERVED_DAYS).any():
    insufficient = candidate_pool.loc[candidate_pool.pretest_observed_days.lt(MIN_PRETEST_OBSERVED_DAYS), "household_id"].tolist()
    raise RuntimeError(f"Eligible model-ready households lack {MIN_PRETEST_OBSERVED_DAYS} pre-test observed days: {insufficient[:10]}")
if candidate_pool["household_id"].isin(natural_unseen_households).any():
    raise RuntimeError("Naturally unseen test households must remain outside the intentional holdout candidate pool")

candidate_pool = candidate_pool.sort_values("household_id").reset_index(drop=True)
candidate_pool["history_length_band"] = pd.qcut(
    candidate_pool["pretest_observed_days"].rank(method="first"), q=3,
    labels=["short", "medium", "long"])
candidate_pool["stratum"] = candidate_pool["predominant_branch"] + " | " + candidate_pool["history_length_band"].astype(str)

eligibility_summary = pd.DataFrame([
    {"population": "all_master_households", "households": master.household_id.nunique()},
    {"population": "outer_test_households", "households": outer_test.household_id.nunique()},
    {"population": "natural_unseen_outer_test", "households": len(natural_unseen_households)},
    {"population": "eligible_seen_candidate_pool", "households": len(candidate_pool)},
])
display(eligibility_summary)
print("Eligible pre-test history days:", candidate_pool.pretest_observed_days.describe(percentiles=[.25, .5, .75, .9]).to_dict())


## Stage 2 — freeze the stratified household assignment

Exactly 20% of eligible household IDs are selected using a fixed seed. The count is allocated proportionally across branch-by-history strata using a largest-remainder rule, then sampled within each stratum. The household list is sorted before sampling so the result is reproducible.


In [ ]:
stratum_counts = candidate_pool.groupby("stratum", as_index=False, observed=True).size().rename(columns={"size": "eligible_households"})
stratum_counts["raw_holdout_count"] = stratum_counts["eligible_households"] * HOLDOUT_FRACTION
stratum_counts["holdout_count"] = np.floor(stratum_counts["raw_holdout_count"]).astype(int)
target_holdout_count = int(round(len(candidate_pool) * HOLDOUT_FRACTION))
remaining = target_holdout_count - int(stratum_counts["holdout_count"].sum())
stratum_counts["remainder_fraction"] = stratum_counts["raw_holdout_count"] - stratum_counts["holdout_count"]
stratum_counts = stratum_counts.sort_values(["remainder_fraction", "stratum"], ascending=[False, True]).reset_index(drop=True)
if remaining < 0 or remaining > len(stratum_counts):
    raise RuntimeError("Largest-remainder household allocation produced an invalid residual")
if remaining:
    stratum_counts.loc[:remaining - 1, "holdout_count"] += 1
stratum_counts = stratum_counts.sort_values("stratum").reset_index(drop=True)
allocation = stratum_counts.set_index("stratum")["holdout_count"].to_dict()

rng = np.random.default_rng(RANDOM_SEED)
holdout_ids = set()
for stratum in sorted(allocation):
    ids = candidate_pool.loc[candidate_pool.stratum.eq(stratum), "household_id"].astype(str).sort_values().to_numpy()
    n_selected = int(allocation[stratum])
    if n_selected:
        holdout_ids.update(rng.choice(ids, size=n_selected, replace=False).tolist())
if len(holdout_ids) != target_holdout_count:
    raise RuntimeError(f"Selected {len(holdout_ids)} households; expected {target_holdout_count}")

candidate_pool["holdout_group"] = np.where(candidate_pool.household_id.astype(str).isin(holdout_ids),
                                             "intentional_holdout", "development_pool")
all_households = pd.Index(master.household_id.dropna().astype(str).unique(), name="household_id")
assignment = pd.DataFrame({"household_id": all_households.astype("string")})
assignment = assignment.merge(candidate_pool[["household_id", "holdout_group"]], on="household_id", how="left", validate="one_to_one")
natural_ids = set(master.loc[master.outer_split.eq("test") & master.test_cohort.eq("test_unseen"), "household_id"].astype(str))
assignment.loc[assignment.household_id.astype(str).isin(natural_ids), "holdout_group"] = "natural_unseen_test"
assignment["holdout_group"] = assignment["holdout_group"].fillna("development_only")
assignment["household_id"] = assignment["household_id"].astype("string")
assignment["holdout_group"] = assignment["holdout_group"].astype("string")
assignment = assignment.sort_values("household_id").reset_index(drop=True)

stratum_audit = candidate_pool.groupby(["predominant_branch", "history_length_band", "stratum", "holdout_group"],
                                       observed=True, as_index=False).agg(
    households=("household_id", "nunique"), median_pretest_observed_days=("pretest_observed_days", "median"))
stratum_pivot = candidate_pool.pivot_table(index=["predominant_branch", "history_length_band", "stratum"],
    columns="holdout_group", values="household_id", aggfunc="nunique", fill_value=0, observed=True).reset_index()
stratum_audit.to_csv(TABLE_DIR / "04b_02_stratum_assignment_audit.csv", index=False, encoding="utf-8")
stratum_pivot.to_csv(TABLE_DIR / "04b_03_stratum_counts.csv", index=False, encoding="utf-8")

ASSIGNMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary = ASSIGNMENT_PATH.with_name(ASSIGNMENT_PATH.stem + ".tmp" + ASSIGNMENT_PATH.suffix)
assignment[["household_id", "holdout_group"]].to_csv(temporary, index=False, encoding="utf-8", na_rep="")
temporary.replace(ASSIGNMENT_PATH)
assignment_contract = write_dtype_contract(assignment[["household_id", "holdout_group"]], ASSIGNMENT_PATH, ASSIGNMENT_CONTRACT_PATH)
print(f"Frozen {len(holdout_ids):,} intentional household holdouts from {len(candidate_pool):,} eligible IDs ({len(holdout_ids)/len(candidate_pool):.2%}).")
display(stratum_pivot)


## Stage 3 — audit test support after assignment

Only after household IDs are selected do we join future P95/P90 labels to summarize available evaluation support. These outcome counts do not change the assignment. The naturally unseen cohort is reported separately from the intentional holdout group.


In [ ]:
# Load future labels only after the household assignment has been written and frozen.
TARGET_KEYS = ["row_id", "household_id", "measurement_date", "target"]
p95_targets = read_contract_csv(P95_PATH, P95_CONTRACT_PATH, TARGET_KEYS)
p90_targets = read_contract_csv(P90_PATH, P90_CONTRACT_PATH, TARGET_KEYS)
for frame in (p95_targets, p90_targets):
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()

assignment_map = assignment.set_index("household_id")["holdout_group"]
test_assignments = outer_test[["row_id", "household_id", "target_date", "test_cohort"]].copy()
test_assignments["holdout_group"] = test_assignments.household_id.map(assignment_map)
if test_assignments["holdout_group"].isna().any():
    raise RuntimeError("One or more outer-test households are missing a frozen assignment")

def test_support(frame, target):
    data = frame[["row_id", "household_id", "measurement_date", "target"]].merge(
        test_assignments[["row_id", "holdout_group", "test_cohort"]], on="row_id", how="inner", validate="one_to_one")
    return data.groupby("holdout_group", as_index=False).agg(
        target= ("target", lambda values: target), rows=("row_id", "size"),
        households=("household_id", "nunique"), unique_dates=("measurement_date", "nunique"),
        positive_events=("target", "sum"), positive_rate=("target", "mean"))

p95_test_support = test_support(p95_targets, "P95")
p90_test_support = test_support(p90_targets, "P90")
test_support_summary = pd.concat([p95_test_support, p90_test_support], ignore_index=True)
# Keep reporting order stable, including groups that have no outer-test rows.
expected_groups = ["intentional_holdout", "development_pool", "natural_unseen_test", "development_only"]
test_support_summary["holdout_group"] = pd.Categorical(test_support_summary.holdout_group, categories=expected_groups, ordered=True)
test_support_summary = test_support_summary.sort_values(["target", "holdout_group"]).reset_index(drop=True)
test_support_summary["holdout_group"] = test_support_summary.holdout_group.astype(str)
test_support_summary.to_csv(TABLE_DIR / "04b_04_test_support_audit.csv", index=False, encoding="utf-8")
display(test_support_summary)

branch_counts = candidate_pool.groupby(["predominant_branch", "holdout_group"], observed=True).size().unstack(fill_value=0)
branch_counts = branch_counts.reindex(columns=["development_pool", "intentional_holdout"], fill_value=0)
branch_share = branch_counts.div(branch_counts.sum(axis=1), axis=0)
ax = branch_share.plot(kind="bar", stacked=True, figsize=(11, 5), color=["#4c78a8", "#f58518"])
ax.set_title("Eligible household assignment share by pre-test branch")
ax.set_xlabel("Predominant branch before outer test")
ax.set_ylabel("Share of households")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
figure_path = FIGURE_DIR / "04b_01_holdout_assignment_by_branch.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Holdout assignment figure:", figure_path)


## Stage 4 — final validation and frozen handoff

The saved assignment includes every household represented in the master split. Eligible households are assigned to `intentional_holdout` or `development_pool`; naturally unseen test households and development-only households have explicit labels for safe downstream joins. Model training notebooks should exclude `intentional_holdout` IDs from all fitting, calibration, and threshold-selection stages.


In [ ]:
candidate_assignment = candidate_pool[["household_id", "holdout_group"]]
intentional_test_support = test_support_summary.loc[
    test_support_summary.holdout_group.eq("intentional_holdout") & test_support_summary.target.isin(["P95", "P90"])
]
all_household_counts = assignment.holdout_group.value_counts()
final_checks = pd.DataFrame([
    ("master_split_matches_frozen_manifest", sha256(MASTER_PATH) == master_signature["sha256"], master_signature["sha256"]),
    ("assignment_has_one_row_per_master_household", assignment.household_id.is_unique and len(assignment) == master.household_id.nunique(), f"{len(assignment):,}/{master.household_id.nunique():,}"),
    ("eligible_pool_is_seen_future_households", set(candidate_pool.household_id.astype(str)) == development_households & test_seen_households, f"{len(candidate_pool):,}"),
    ("eligible_pool_has_minimum_pretest_history", candidate_pool.pretest_observed_days.ge(MIN_PRETEST_OBSERVED_DAYS).all(), str(int(candidate_pool.pretest_observed_days.min()))),
    ("intentional_holdout_fraction_near_20_percent", len(holdout_ids) == target_holdout_count and abs(len(holdout_ids)/len(candidate_pool)-HOLDOUT_FRACTION) <= 1/len(candidate_pool), f"{len(holdout_ids)}/{len(candidate_pool)}={len(holdout_ids)/len(candidate_pool):.4%}"),
    ("assignment_groups_are_disjoint_and_complete", set(assignment.holdout_group.dropna().unique()).issubset({"intentional_holdout", "development_pool", "natural_unseen_test", "development_only"}) and assignment.holdout_group.notna().all(), str(all_household_counts.to_dict())),
    ("natural_unseen_cohort_remains_separate", set(assignment.loc[assignment.holdout_group.eq("natural_unseen_test"), "household_id"].astype(str)) == natural_unseen_households, f"{len(natural_unseen_households)} households"),
    ("holdout_group_sampled_without_replacement", len(holdout_ids) == len(set(holdout_ids)) and not (holdout_ids & (development_households - set(candidate_pool.household_id.astype(str)))), f"seed={RANDOM_SEED}"),
    ("holdout_P95_test_event_support", int(intentional_test_support.loc[intentional_test_support.target.eq("P95"), "positive_events"].sum()) >= MIN_HOLDOUT_TEST_EVENTS, str(intentional_test_support.set_index("target").positive_events.to_dict())),
    ("holdout_P90_test_event_support", int(intentional_test_support.loc[intentional_test_support.target.eq("P90"), "positive_events"].sum()) >= MIN_HOLDOUT_TEST_EVENTS, str(intentional_test_support.set_index("target").positive_events.to_dict())),
    ("assignment_csv_has_required_columns", pd.read_csv(ASSIGNMENT_PATH, nrows=0).columns.tolist() == ["household_id", "holdout_group"], str(ASSIGNMENT_PATH)),
    ("assignment_dtype_contract_exists", ASSIGNMENT_CONTRACT_PATH.exists(), str(ASSIGNMENT_CONTRACT_PATH)),
    ("stratum_audit_and_figure_exist", (TABLE_DIR / "04b_02_stratum_assignment_audit.csv").exists() and figure_path.exists() and figure_path.stat().st_size > 0, str(figure_path)),
], columns=["check", "passed", "evidence"])
final_checks.to_csv(TABLE_DIR / "04b_05_final_validation.csv", index=False, encoding="utf-8")
display(final_checks)
if not final_checks.passed.all():
    raise RuntimeError(f"Unseen-household design validation failed: {final_checks.loc[~final_checks.passed, 'check'].tolist()}")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "04b_unseen_household_holdout_design",
    "input_signatures": {str(path): {"sha256": sha256(path), "size_bytes": path.stat().st_size}
                         for path in [MASTER_PATH, P95_PATH, P90_PATH, OBSERVED_PATH]},
    "master_split_manifest": str(SPLIT_MANIFEST_PATH),
    "outer_test_start": str(TEST_START.date()),
    "eligible_candidate_definition": "household has outer-development and test_seen rows in the frozen master split; test rows are model-ready",
    "eligible_households": len(candidate_pool),
    "holdout_fraction_target": HOLDOUT_FRACTION,
    "intentional_holdout_households": len(holdout_ids),
    "intentional_holdout_fraction_observed": len(holdout_ids) / len(candidate_pool),
    "random_seed": RANDOM_SEED,
    "stratification_fields": ["predominant_branch_before_outer_test", "pretest_observed_history_length_tertile"],
    "pretest_history_minimum_observed_days": MIN_PRETEST_OBSERVED_DAYS,
    "future_target_outcomes_used_for_assignment": False,
    "holdout_group_values": {key: int(value) for key, value in all_household_counts.items()},
    "assignment": {"path": str(ASSIGNMENT_PATH), "rows": len(assignment), "columns": ["household_id", "holdout_group"], "sha256": sha256(ASSIGNMENT_PATH)},
    "assignment_dtype_contract": str(ASSIGNMENT_CONTRACT_PATH),
    "validation": {str(row.check): bool(row.passed) for row in final_checks.itertuples(index=False)},
    "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
write_json(manifest, TABLE_DIR / "04b_unseen_household_holdout_manifest.json")
print("Unseen-household assignment frozen; all checks passed.")
print(f"Intentional holdout: {len(holdout_ids):,} of {len(candidate_pool):,} eligible households.")
print("Assignment CSV:", ASSIGNMENT_PATH)


### Downstream use

Join `04b_household_holdout_assignment.csv` by `household_id`. In the later generalization experiment, remove `intentional_holdout` households from classifier fitting, calibration, and threshold selection, then evaluate their outer-test rows. Keep `natural_unseen_test` as a separate exploratory cohort.
